# Notebook 01 — `INNER JOIN`, `SELF JOIN`, and Multi-Table Querying

**Session:** Database - SQL (2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Load the relational enterprise schema from `../data/sql_store_seed.sql` into an in-memory SQLite database and master:
1. `INNER JOIN` with explicit `ON` predicates and table aliases.
2. Cross-table projections and computed line-item values.
3. Recursive `SELF JOIN` queries on employee-manager organizational hierarchies.
4. Chaining 3+ tables across order management and commercial invoicing pipelines.
5. Compound join conditions on composite keys `(order_id, product_id)`.
6. Explicit ANSI SQL-92 joins vs. legacy implicit comma joins, and analyzing accidental Cartesian cross products.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Loaded tables in database:")
show_rows(query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;"))

## 1. Two-Table `INNER JOIN` and Computed Columns

An `INNER JOIN` evaluates the Cartesian product of two relations and filters for rows satisfying the equi-join predicate. Table aliases keep references concise and prevent ambiguous column collisions.

In [ ]:
# Basic inner join between orders and customers
rows = query("""
SELECT
    o.order_id,
    o.order_date,
    c.first_name,
    c.last_name
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")
print(f"Total matched orders: {len(rows)}")
show_rows(rows, limit=5)
assert len(rows) == 14
assert rows[0]["order_id"] == 1
assert rows[0]["first_name"] == "Elka"

In [ ]:
# Joining order_items with products to compute item line subtotals
item_rows = query("""
SELECT
    oi.order_id,
    oi.product_id,
    p.name AS product_name,
    oi.quantity,
    oi.unit_price,
    ROUND(oi.quantity * oi.unit_price, 2) AS line_subtotal
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
ORDER BY oi.order_id, oi.product_id;
""")
print(f"Total order line items: {len(item_rows)}")
show_rows(item_rows, limit=6)
assert len(item_rows) == 22
assert item_rows[0]["product_name"] == "Brocolinni - Gaylan, Chinese"
assert item_rows[0]["line_subtotal"] == 14.96

## 2. Recursive Hierarchies: The `SELF JOIN`

In the `employees` table, the `reports_to` foreign key points back to `employee_id` in the very same table. By joining `employees` to itself with distinct aliases (`e` for employee, `m` for manager), we resolve the supervisory chain.

In [ ]:
self_join_rows = query("""
SELECT
    e.employee_id,
    e.first_name || ' ' || e.last_name AS employee_name,
    e.job_title AS employee_title,
    m.employee_id AS manager_id,
    m.first_name || ' ' || m.last_name AS manager_name,
    m.job_title AS manager_title
FROM employees e
JOIN employees m
    ON e.reports_to = m.employee_id
ORDER BY m.employee_id, e.employee_id;
""")
print(f"Employees with an assigned manager: {len(self_join_rows)}")
show_rows(self_join_rows)

# Note: The EVP (Yovonnda Maggard) has reports_to = NULL and is excluded by INNER JOIN
assert len(self_join_rows) == 7
assert any(
    r["employee_name"] == "Darcy Northey" and r["manager_name"] == "Yovonnda Maggard"
    for r in self_join_rows
)
assert any(
    r["employee_name"] == "Keriann Alloisi" and r["manager_name"] == "Sayer Gullis"
    for r in self_join_rows
)

## 3. Chaining Multiple Tables (3+ Way Joins)

Transactional queries often traverse several tables in a single operation. Here we inspect two real-world multi-table pipelines:
1. **Order Fulfillment:** `orders` + `customers` + `order_statuses`
2. **Invoicing & Cashflow:** `payments` + `clients` + `invoices` + `payment_methods`

In [ ]:
# 3-way join: Orders with customer info and order status name
order_pipeline = query("""
SELECT
    o.order_id,
    o.order_date,
    c.first_name || ' ' || c.last_name AS customer_name,
    c.state,
    s.name AS status_name
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_statuses s
    ON o.status = s.order_status_id
ORDER BY o.order_id;
""")
print(f"Order pipeline rows: {len(order_pipeline)}")
show_rows(order_pipeline, limit=6)
assert len(order_pipeline) == 14
assert order_pipeline[0]["status_name"] == "Processed"
assert order_pipeline[1]["status_name"] == "Shipped"

In [ ]:
# 4-way join: Complete client payment audit trail
invoicing_pipeline = query("""
SELECT
    p.payment_id,
    p.date AS payment_date,
    p.amount,
    c.name AS client_name,
    c.phone AS client_phone,
    i.number AS invoice_number,
    i.invoice_total,
    i.payment_total,
    pm.name AS payment_method
FROM payments p
JOIN clients c
    ON p.client_id = c.client_id
JOIN invoices i
    ON p.invoice_id = i.invoice_id
JOIN payment_methods pm
    ON p.payment_method = pm.payment_method_id
ORDER BY p.payment_id;
""")
print(f"Payment receipts audited: {len(invoicing_pipeline)}")
show_rows(invoicing_pipeline)
assert len(invoicing_pipeline) == 8
assert invoicing_pipeline[0]["client_name"] == "Topico Systems"
assert invoicing_pipeline[0]["payment_method"] == "Credit Card"

## 4. Compound Join Conditions (Composite Primary & Foreign Keys)

The `order_items` table is uniquely identified by `(order_id, product_id)`. When joining to `order_item_notes`, the join predicate must assert equality across **both** keys simultaneously.

In [ ]:
compound_rows = query("""
SELECT
    oi.order_id,
    oi.product_id,
    p.name AS product_name,
    oi.quantity,
    oin.note AS packaging_instructions
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN order_item_notes oin
    ON oi.order_id = oin.order_id
   AND oi.product_id = oin.product_id
ORDER BY oi.order_id, oi.product_id;
""")
print(f"Annotated line items: {len(compound_rows)}")
show_rows(compound_rows)
assert len(compound_rows) == 8
assert compound_rows[0]["packaging_instructions"] == "Fragile - pack with double bubble wrap"

## 5. Explicit vs. Implicit Joins and the Accidental Cross Join Danger

Compare modern explicit ANSI SQL-92 joins with legacy implicit comma joins. Notice the danger: if an engineer forgets the `WHERE` clause in an implicit join, the query silently produces an unintentional Cartesian product (`CROSS JOIN`).

In [ ]:
# ANSI SQL-92 Explicit Join
explicit_rows = query("""
SELECT o.order_id, c.last_name
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")

# ANSI SQL-89 Implicit Join
implicit_rows = query("""
SELECT o.order_id, c.last_name
FROM orders o, customers c
WHERE o.customer_id = c.customer_id
ORDER BY o.order_id;
""")
assert explicit_rows == implicit_rows
print(f"Both forms return identical {len(explicit_rows)} matched rows.")

# Dangerous bug: Forgetting the WHERE clause!
accidental_cartesian = query("SELECT COUNT(*) AS total_rows FROM orders o, customers c;")
num_orders = query("SELECT COUNT(*) AS c FROM orders;")[0]["c"]
num_customers = query("SELECT COUNT(*) AS c FROM customers;")[0]["c"]
print(
    f"Accidental Cartesian product: {num_orders} * {num_customers} = {accidental_cartesian[0]['total_rows']} rows!"
)
assert accidental_cartesian[0]["total_rows"] == num_orders * num_customers

# Legitimate deliberate CROSS JOIN: Product catalog by shipping carrier matrix
cross_rows = query("""
SELECT p.name AS product_name, s.name AS shipper_name
FROM products p
CROSS JOIN shippers s
ORDER BY p.name, s.name;
""")
print(f"Deliberate cross join catalog matrix size: {len(cross_rows)}")
show_rows(cross_rows, limit=5)
assert len(cross_rows) == 13 * 5